# Operator Compiler Tutorial

Welcome to the **Operator Compiler Tutorial**!

This interactive notebook will guide you through the fundamentals of **performance engineering** using the XTC compiler, a research platform for optimizing AI operators.

By the end of this notebook, you will understand how to:

- Define computational graphs with XTC
- Compile and evaluate operator performance
- Explore the scheduling space to find optimal configurations

## Define your first graph

In XTC, computations are represented as **dataflow graphs**. A graph consists of:
- **Nodes** representing operations each implementing a specific computation (Operators)
- **Edges** representing data dependencies between operations (through Tensors)

Where:
- **Tensors** are multi-dimensional arrays that hold data
- **Operators** are tensor operations (e.g., matrix multiplication, convolution)

Let us start by creating a simple matrix multiplication graph. Matrix multiplication (matmul) computes $C = A \times B$ where:
- $A$ is an $I \times K$ matrix
- $B$ is a $K \times J$ matrix
- $C$ is the resulting $I \times J$ matrix

**Practice.** The code below is editable and the output (i.e. the serialized graph) is dynamically computed.

1. Try modifying the dimensions or data type (float32, float64) to see how the graph changes!
2. Add a ReLU activation after the matrix multiplication. Hint: `O.matmul()` returns a tensor that can be passed to another operator. Using ReLu: `R = O.relu(inp, name)`.

In [1]:
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

I, J, K, dtype = 16, 1000, 512, "float32"
graph = matmul_graph(I=I,J=J,K=K,dtype=dtype)

print(graph)

graph:
  name: matmul
  inputs:
  - %0 : 16x512xfloat32
  - %1 : 512x1000xfloat32
  outputs:
  - %2 : 16x1000xfloat32
  nodes:
  - %2: matmul(%0, %1) {name = 'C'} : [16x512xfloat32, 512x1000xfloat32] -> [16x1000xfloat32]



## Compile and Evaluate

Now that we have a graph, we can compile it and measure its baseline performance (without any optimization).

The compilation pipeline in XTC follows these steps:
1. **Create a Backend**: In XTC, the backend corresponds to an existing framework such as MLIR or TVM that, given a schedule, can generate the code for a specific target
2. **Get a Scheduler**: In XTC, a scheduler is a builder that creates a schedule. Even without optimizations, we need a scheduler to get a default loop structure.
3. **Compile**: Generate executable code
4. **Evaluate**: Run the compiled code and measure performance

**Practice.** The script below compiles the matmul graph without any optimization and displays both the generated code (if you unroll the accordion) and its performance on chip.

1. Use the radio buttons to select optionally an output (IR, generated assembly or equivalent optimized C code)
2. *Inspect the generated code.* Look at the Source IR, Transformed IR, Assembly and C. Are you able to identify the different loops i, j, k ?
3. *Observe the performance.* In your opinion, why is it so poor?

**Note:** Performance is measured as a percentage of peak (the estimated FLOP/s of one CPU core). The latter is by automatically approximated by running a benchmark, but this technique comes with (little) noise.

In [2]:
import ipy
ipy.display_nradio("print", "Print options for the execution of next cell", "none", "source IR", "transformed IR", "Assembly", "C")

RadioButtons(description='Print options for the execution of next cell', options=(('none', 'none'), ('source I…

In [3]:
import ipy
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_naive"

graph = matmul_graph(I, J, K, dtype)

# Compile (no transformations, just default loop structure)
backend = Backend(graph)
scheduler = backend.get_scheduler()
schedule = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source IR"),
    print_transformed_ir=ipy.nradio_value("print", "transformed IR"),
    print_assembly=ipy.nradio_value("print", "Assembly"),
    emit_c=ipy.nradio_value("print", "C"),
)
compiler = backend.get_compiler(dump_file=base, shared_lib=True, **print_opts)
module = compiler.compile(schedule)

if ipy.nradio_value("print", "C"):
    print(Path(f"{base}.c").read_text())

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")


Peak perfromance: 1.68 %


## Optimize with Scheduling

The baseline compilation produces correct but unoptimized code. To improve performance, XTC exposes a **scheduler** with imperative primitives that transform the loop nest:

- `sch.interchange(["i", "k", "j"])` reorders the loops. Interchange improves memory access patterns by ensuring stride-1 access (contiguous memory) rather than strided access, maximizing cache efficiency. Along with primitive `sch.strip_mine` (see below) this allows to actually tile the loop body.
- `sch.strip_mine("j", {"j1": 16})` breaks loop `j` into smaller chunks of size 16. This transformation can also be seen as 1D tiling.
- To perform actual multi-dimensional tiling, combine `sch.strip_mine` with `sch.interchange`. For example, 2D tiling:
  ```python
  sch.strip_mine("j", {"j1": 16})
  sch.strip_mine("k", {"k1": 16})
  sch.interchange(["i", "j", "k", "j1", "k1"])
  ```
  This creates $16 \times 16$ tiles over the $(j, k)$ dimensions.
- `sch.vectorize(["j1"])` vectorizes the computation along the loop `j1`. Vectorization uses SIMD instructions to process multiple elements in parallel, significantly increasing throughput on modern CPUs.
- `sch.unroll({"j1":1})` unrolls the loop `j1` with an unroll factor of 1 (which has no effect). Unrolling reduces loop overhead (fewer branches) and exposes more instruction-level parallelism to the hardware.
- `sch.parallelize(["i"])` execute in parallel along the loop `i`. Parallelization is only useful on a multiple cores architecture and actually splits and dispatch the work onto the available cores.
- `sch.split("j", {"j0": 0, "j1": 16})` splits `j` into two segments, creating `j0` (iterations 0-15) and `j1` (iterations 16+). Splitting is useful for applying different transformations to different parts of a loop (e.g., vectorize the main part, handle the remainder separately).

**Practice.** The code below lets you define a schedule using imperative primitives. Use the radio buttons to select the backend and what IR to display. Compare the performance and generated code with the unoptimized version from the previous section.

1. *Transform the code.* Start with simple transformations and build up.
2. *Inspect the generated code.* How does the Assembly or IR differ from the baseline?
3. *Try to maximize the performance!*

*Hint: You may want to create a register tile (i1,j1), j1 being a multiple of your SIMD registers size. Generally 8.*

In [4]:
import ipy
ipy.display_nradio("print", "Print options for the execution of next cell", "none", "source IR", "transformed IR", "Assembly", "C")

RadioButtons(description='Print options for the execution of next cell', options=(('none', 'none'), ('source I…

In [5]:
import ipy
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

# Schedule definition
def schedule(sch):
    """Apply transformations to the scheduler."""
    sch.set_dims(['i','j','k'])
    # Add transformations here, e.g.:
    # sch.strip_mine("j", {"j1": 4})
    # sch.vectorize(["j1"])
    # sch.interchange(["i", "j", "k", "j1"])

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_opt"

graph = matmul_graph(I, J, K, dtype)

# Compile
backend = Backend(graph)
scheduler = backend.get_scheduler()
schedule(scheduler)
sched = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source IR"),
    print_transformed_ir=ipy.nradio_value("print", "transformed IR"),
    print_assembly=ipy.nradio_value("print", "Assembly"),
    emit_c=ipy.nradio_value("print", "C"),
)
compiler = backend.get_compiler(dump_file=base, shared_lib=True, **print_opts)
module = compiler.compile(sched)

if ipy.nradio_value("print", "C"):
    print(Path(f"{base}.c").read_text())

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")


Peak perfromance: 1.77 %


## Define a schedule declaratively

XTC allows you to describe the target loop structure using a Python dictionary. Instead of manually specifying each transformation step, you declare the desired final loop structure, and XTC automatically infers the sequence of transformations needed to achieve it.

For example, the following loop structure:
```python
for i in ...
  for k in ...
    for j in ...
      for j1 in range(16):  // vectorized
```

Can be described as:
```python
{
  "i": {},
    "k": {},
      "j": {},
        "j#16": {"vectorize": True}
}
```

The dictionary keys define the loop order (outer to inner), `j#16` creates a tile of size 16 on `j`, and the `{"vectorize": True}` attribute marks that inner loop for vectorization.

The declarative API supports several key optimizations:

| Transformation      | Syntax                          |
|---------------------|---------------------------------|
| **Tiling**          | `"axis#size"`                   |
| **Vectorization**   | `{"vectorize": True}`           |
| **Parallelization** | `{"parallelize": True}`         |
| **Unrolling**       | `{"unroll": factor}`            |
| **Interchange**     | Key ordering in the dictionnary |
| **Splitting**       | `"axis[beg:end]"`               |

**Practice.** The code below lets you define a schedule specification and see the generated assembly. Try replicating the good-enough schedule you discovered in the previous section!

In [6]:
import ipy
ipy.display_nradio("print", "Print options for the execution of next cell", "none", "source IR", "transformed IR", "Assembly", "C")

RadioButtons(description='Print options for the execution of next cell', options=(('none', 'none'), ('source I…

In [7]:
import ipy
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from xtc.schedules.descript import descript_scheduler
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

# Schedule definition
def schedule_declarative(sch):
    """Define here your declarative scheduler."""
    schedule_spec = {
        # Add additional axes and directives below
        "i": {},
        "j": {},
        "k": {},
        "j#4":{ "vectorize": True},
    }

    descript_scheduler(
        scheduler=scheduler,
        node_name="C",
        abstract_dims=["i", "j", "k"],
        spec=schedule_spec
    )

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_desc"

graph = matmul_graph(I, J, K, dtype)

# Compile
backend = Backend(graph)
scheduler = backend.get_scheduler()
schedule_declarative(scheduler)
sched = scheduler.schedule()

print_opts = dict(
    print_source_ir=ipy.nradio_value("print", "source IR"),
    print_transformed_ir=ipy.nradio_value("print", "transformed IR"),
    print_assembly=ipy.nradio_value("print", "Assembly"),
    emit_c=ipy.nradio_value("print", "C"),
)
compiler = backend.get_compiler(dump_file=base, shared_lib=True, **print_opts)
module = compiler.compile(sched)

if ipy.nradio_value("print", "C"):
    print(Path(f"{base}.c").read_text())

# Evaluate and display results
peak_flops = HostRuntime.get().evaluate_flops(dtype)
evaluator = module.get_evaluator()
results, _, _ = evaluator.evaluate()
perf = (I * J * K) / min(results) / peak_flops * 100

print(f"\nPeak perfromance: {perf:.2f} %")


Peak perfromance: 7.28 %


## Experimenting with Multiple Schedules

Performance engineering is often about exploring different optimization strategies. Different schedules can have dramatically different performance depending on:
- **Problem size**: Small matrices may not benefit from parallelization overhead
- **Hardware**: Cache sizes, vector width, and core count affect optimal tiling
- **Data layout**: Memory access patterns influence cache efficiency

In this section, we'll write a simple loop to try several schedule configurations and compare their performance.

**Practice.** The code below defines several schedule configurations using our declarative scheduler. You can:

- **Add/modify configurations**: Try different tile sizes, loop orderings, or optimization combinations. The pre-built search space (variable `configurations` in function `explore` and lines above) may be poorly designed...
- **Change the acquisition function**: Add caching, error handling, or custom metrics
- **Modify the exploration loop**: Add early stopping or custom filtering

The `explore()` function must `yield` tuples of `(index, total, config_name, performance)` for real-time progress display.

In [8]:
import ipy
import itertools
from tqdm.notebook import tqdm
import xtc.graphs.xtc.op as O
from xtc.graphs.xtc.graph import XTCGraph
from xtc.backends.tvm import Backend
from xtc.runtimes.host import HostRuntime
from xtc.schedules.descript import descript_scheduler
from pathlib import Path

def matmul_graph(I: int, J: int, K: int, dtype: str) -> XTCGraph:
   """Create a graph computing C = A @ B."""
   a = O.tensor((I, K), dtype, name="A")
   b = O.tensor((K, J), dtype, name="B")
   with O.graph(name="matmul") as gb:
     C = O.matmul(a, b, name="C")
   return gb.graph

def apply_schedule(graph, spec):
    """Apply a declarative schedule specification and compile."""
    backend = Backend(graph)
    scheduler = backend.get_scheduler()
    descript_scheduler(
        scheduler=scheduler,
        node_name="C",
        abstract_dims=["i", "j", "k"],
        spec=spec
    )
    schedule = scheduler.schedule()
    comp = backend.get_compiler(dump_file="matmul_explore", shared_lib=True)
    module = comp.compile(schedule)
    return module

def evaluate(module, peak_flops, nfmadds):
    """Evaluate module performance as percentage of peak."""
    evaluator = module.get_evaluator()
    results, _, _ = evaluator.evaluate()
    result = min(results)
    time_flops = nfmadds / result
    perf = time_flops / peak_flops * 100
    return perf

def acquire(*params):
    """Evaluate a single configuration given the params and return its performance."""
    schedule_spec = schedule_declarative(*params)
    module = apply_schedule(
        graph=graph,
        spec=schedule_spec,
    )
    return evaluate(module, peak_flops, I*J*K)

# Schedule definition (parametric), add any required parameters
def schedule_declarative(i1, j1):
    """Define here your declarative scheduler."""
    schedule_spec = {
        # Add additional axes and directives below
        "i": {},
        "j": {},
        "k": {},
        f"i#{i1}": {"unroll": True},
        f"j#{j1}": {"vectorize": True}
    }
    return schedule_spec

# Exploration generator
def explore():
   """Generator that yields (index, total, name, perf) for each evaluation."""
   i1_range = range(1, 5)
   j1_range = range(1, 5)
   configurations = list(itertools.product(i1_range, j1_range))
   total = len(configurations)

   for idx, (i1, j1) in enumerate(configurations):
     perf = acquire(i1, j1)
     yield (idx, total, f"{base}:{i1}x{j1}", perf)

# Problem setup
I, J, K, dtype = 16, 1000, 512, "float32"
base = "matmul_expl"

graph = matmul_graph(I, J, K, dtype)
peak_flops = HostRuntime.get().evaluate_flops(dtype)

results = []
best_perf = 0
progress = tqdm(explore())
for idx, total, name, perf in progress:
    best_perf = max(best_perf, perf)
    progress.total = total
    progress.set_description(f"Best: {best_perf:.2f}%, current: {perf:.2f}%")
    results.append((idx, total, name, perf))

best = sorted(results, key=lambda x: -x[3])[0]
print(f"Found best performance for {best[2]} at peak perf: {best[3]:.2f}%")

0it [00:00, ?it/s]

Found best performance for matmul_expl:4x4 at peak perf: 25.23%
